# BÀI TẬP: TITANIC
**Nguồn:** kaggle.com/c/titanic (891 dòng)


In [1]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path
from scipy import stats

sns.set_style('whitegrid')

csv_path = 'https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv'

df = pd.read_csv(csv_path)
print('Loaded from:', csv_path)
df.head()

Loaded from: https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


---
# PHẦN A — DATA PROFILING
## A.1. Data size, column names, data types

In [2]:
df.shape
df.columns.tolist()
df.dtypes

survived         int64
pclass           int64
sex             object
age            float64
sibsp            int64
parch            int64
fare           float64
embarked        object
class           object
who             object
adult_male        bool
deck            object
embark_town     object
alive           object
alone             bool
dtype: object

## A.2. Missing values & Duplicate data

In [3]:
df.isna().sum()
df.duplicated().sum()

np.int64(107)

## A.3. Invalid values

In [4]:
df[(df['survived'].isin([0, 1]) == False) | (df['pclass'].isin([1, 2, 3]) == False) | (df['age'] < 0) | (df['sibsp'] < 0) | (df['parch'] < 0) | (df['fare'] < 0)]

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone


## A.4. Create a new column
Tạo cột `family_size` = sibsp + parch + 1.

In [5]:
df['family_size'] = df['sibsp'] + df['parch'] + 1

---
# PHẦN B — DESCRIPTIVE STATISTICS
## Group 1 — Central Tendency

In [6]:
numeric_df = df.select_dtypes(include=np.number)
pd.DataFrame({'mean': numeric_df.mean(), 'median': numeric_df.median(), 'mode': numeric_df.mode().iloc[0]})

,mean,median,mode
survived,0.383838,0.0000,0.00
pclass,2.308642,3.0000,3.00
age,29.699118,28.0000,24.00
sibsp,0.523008,0.0000,0.00
parch,0.381594,0.0000,0.00
fare,32.204208,14.4542,8.05
family_size,1.904602,1.0000,1.00


## Group 2 — Dispersion

In [7]:
pd.DataFrame({'range': numeric_df.max() - numeric_df.min(), 'variance': numeric_df.var(), 'std': numeric_df.std(), 'IQR': numeric_df.quantile(0.75) - numeric_df.quantile(0.25)})

,range,variance,std,IQR
survived,1.0000,0.236772,0.486592,1.0000
pclass,2.0000,0.699015,0.836071,1.0000
age,79.5800,211.019125,14.526497,17.8750
sibsp,8.0000,1.216043,1.102743,1.0000
parch,6.0000,0.649728,0.806057,0.0000
fare,512.3292,2469.436846,49.693429,23.0896
family_size,10.0000,2.603248,1.613459,1.0000


## Group 3 — Location and Shape

In [8]:
pd.DataFrame({'Q1': numeric_df.quantile(0.25), 'median': numeric_df.median(), 'Q3': numeric_df.quantile(0.75), 'skewness': numeric_df.skew(), 'kurtosis': numeric_df.kurt()})

,Q1,median,Q3,skewness,kurtosis
survived,0.0000,0.0000,1.0,0.478523,-1.775005
pclass,2.0000,3.0000,3.0,-0.630548,-1.280015
age,20.1250,28.0000,38.0,0.389108,0.178274
sibsp,0.0000,0.0000,1.0,3.695352,17.880420
parch,0.0000,0.0000,0.0,2.749117,9.778125
fare,7.9104,14.4542,31.0,4.787317,33.398141
family_size,1.0000,1.0000,2.0,2.727441,9.159666


---
# PHẦN C — DEFINE THE QUESTION

## Câu hỏi 1: Hạng vé nào có tỷ lệ sống sót cao nhất, chênh lệch bao nhiêu so với hạng thấp nhất?

In [9]:
class_survival = df.groupby('pclass')['survived'].mean().sort_values(ascending=False)
class_survival.to_frame('survival_rate').assign(survival_percentage=class_survival * 100, gap_from_lowest=class_survival - class_survival.min())

,survival_rate,survival_percentage,gap_from_lowest
pclass,,,
1,0.629630,62.962963,0.387267
2,0.472826,47.282609,0.230464
3,0.242363,24.236253,0.000000


## Câu hỏi 2: Giới tính hay hạng vé ảnh hưởng đến sống sót mạnh hơn?

In [10]:
survival_by_sex = df.groupby('sex')['survived'].mean().sort_values(ascending=False)
survival_by_class = df.groupby('pclass')['survived'].mean().sort_values(ascending=False)
survival_by_sex, survival_by_class, survival_by_sex.max() - survival_by_sex.min(), survival_by_class.max() - survival_by_class.min()

(sex
 female    0.742038
 male      0.188908
 Name: survived, dtype: float64,
 pclass
 1    0.629630
 2    0.472826
 3    0.242363
 Name: survived, dtype: float64,
 np.float64(0.5531300709799203),
 np.float64(0.3872671041713812))

## Câu hỏi 3: Vé đắt hơn có thực sự sống sót cao hơn không?

In [11]:
fare_survival_corr = stats.pointbiserialr(df['survived'], df['fare'])
fare_groups = pd.qcut(df['fare'], q=4, duplicates='drop')
fare_groups_survival = df.groupby(fare_groups, observed=True)['survived'].mean()
fare_survival_corr, fare_groups_survival

(SignificanceResult(statistic=np.float64(0.25730652238496377), pvalue=np.float64(6.1201893419221565e-15)),
 fare
 (-0.001, 7.91]     0.197309
 (7.91, 14.454]     0.303571
 (14.454, 31.0]     0.454955
 (31.0, 512.329]    0.581081
 Name: survived, dtype: float64)

## Câu hỏi 4: Gia đình đông người có ảnh hưởng đến khả năng sống sót không?

In [12]:
df.groupby('family_size')['survived'].agg(['mean', 'count']).sort_index()

,mean,count
family_size,,
1,0.303538,537
2,0.552795,161
3,0.578431,102
4,0.724138,29
5,0.200000,15
6,0.136364,22
7,0.333333,12
8,0.000000,6
11,0.000000,7


## Câu hỏi 5: Cảng lên tàu (embark_town) nào có tỷ lệ sống sót cao nhất?

In [13]:
df.groupby('embark_town')['survived'].agg(['mean', 'count']).sort_values('mean', ascending=False)

,mean,count
embark_town,,
Cherbourg,0.553571,168
Queenstown,0.389610,77
Southampton,0.336957,644


## Câu hỏi 6 (Tổng hợp) — Viết insight tổng hợp
Dựa trên Phần A, B, C, viết 4-5 câu insight tổng thể về dữ liệu Titanic.

Tỷ lệ sống sót chung của hành khách là khoảng 38,4%; hạng 1 có tỷ lệ sống 63,0%, cao hơn hạng 3 khoảng 38,7 điểm phần trăm. Giới tính có sự khác biệt lớn hơn hạng vé, với tỷ lệ sống của nữ là 74,2% và nam là 18,9%. Giá vé có tương quan dương với khả năng sống sót, và tỷ lệ sống tăng từ 19,7% ở nhóm giá vé thấp nhất lên 58,1% ở nhóm cao nhất. Nhóm gia đình nhỏ từ 2 đến 4 người có tỷ lệ sống cao hơn người đi một mình, nhưng các gia đình quá đông có tỷ lệ sống thấp và số quan sát ít. Hành khách lên tàu tại Cherbourg có tỷ lệ sống cao nhất là 55,4%, so với 33,7% tại Southampton.